# CheXSpark — Stage-2: Multi-label Classification (DenseNet121)

Huấn luyện DenseNet121 dự đoán **14 bệnh lý lồng ngực** trên tập dev 2,000 ảnh NIH ChestX-ray14.

**Đính kèm 2 dataset vào notebook này (Add data):**
1. Ảnh 224×224: `khanfashee/nih-chest-x-ray-14-224x224-resized`
2. Nhãn + mask RLE: `<tai-khoan>/chexspark-stage2-nih-dev` (upload từ `data/kaggle/stage2-nih-dev`)

**Pipeline:** đọc ảnh → decode mask phổi CheXmask (RLE) → mask×ảnh (tắt nền ngoài phổi) → DenseNet121 → BCEWithLogitsLoss (pos_weight) → eval AUC/label.

Chọn **GPU** trước khi chạy (Accelerator → GPU T4 x2 / P100).

In [ ]:
# ============ CONFIG — chỉnh ở đây nếu path Kaggle khác ============
import os

IMG_ROOT  = '/kaggle/input/nih-chest-x-ray-14-224x224-resized'  # ảnh 224x224 PNG
META_ROOT = '/kaggle/input/chexspark-stage2-nih-dev'             # stage2_train.csv
TRAIN_CSV = os.path.join(META_ROOT, 'stage2_train.csv')

IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 8
LR          = 1e-4
NUM_WORKERS = 2
MASK_MODE   = 'lungs'      # 'lungs' | 'lungs_heart' | 'none' (baseline không mask)
SEED        = 42

In [ ]:
import os, random, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| torch', torch.__version__)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

In [ ]:
# Xây index {basename: full_path} để join theo image_index (không đoán cấu trúc folder)
print('Quét ảnh trong', IMG_ROOT, '...')
img_index = {}
for dirpath, _, filenames in os.walk(IMG_ROOT):
    for fn in filenames:
        if fn.lower().endswith('.png'):
            img_index[fn] = os.path.join(dirpath, fn)
print(f'Tìm thấy {len(img_index)} PNG')
if len(img_index) == 0:
    print('-> Không thấy PNG loose. Top-level:', os.listdir(IMG_ROOT))
    print('   (nếu ảnh nằm trong .zip, cần giải nén — chỉnh cell này)')

In [ ]:
df = pd.read_csv(TRAIN_CSV, dtype={'image_index': str})
df['path'] = df['image_index'].map(img_index)
found = df['path'].notna().sum()
print(f'Rows: {len(df)} | có ảnh trên Kaggle: {found} ({found/len(df):.1%})')
if found < len(df):
    print('Thiếu ví dụ:', df.loc[df["path"].isna(), 'image_index'].head(5).tolist())
df = df.dropna(subset=['path']).reset_index(drop=True)

LABELS = [c for c in df.columns if c.startswith('label_')]
print('Labels:', len(LABELS), LABELS)

tr_df = df[df.split == 'train'].reset_index(drop=True)
va_df = df[df.split == 'val'].reset_index(drop=True)
te_df = df[df.split == 'test'].reset_index(drop=True)
print(f'train/val/test = {len(tr_df)}/{len(va_df)}/{len(te_df)}')

In [ ]:
def decode_rle(rle, h, w):
    '''CheXmask RLE: chuoi cap (start, length), reshape row-major (C order).'''
    vals = [int(v) for v in str(rle).split()]
    flat = np.zeros(h * w, dtype=np.uint8)
    for i in range(0, len(vals), 2):
        flat[vals[i]:vals[i] + vals[i + 1]] = 1
    return flat.reshape((h, w))


class CheXSparkDataset(Dataset):
    def __init__(self, frame, img_size=224, mask_mode='lungs'):
        self.df = frame
        self.img_size = img_size
        self.mask_mode = mask_mode
        self.norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.df)

    def _mask(self, row):
        h, w = int(row['Height']), int(row['Width'])
        m = decode_rle(row['Left Lung'], h, w) | decode_rle(row['Right Lung'], h, w)
        if self.mask_mode == 'lungs_heart':
            m = m | decode_rle(row['Heart'], h, w)
        m = Image.fromarray((m * 255).astype(np.uint8)).resize(
            (self.img_size, self.img_size), Image.NEAREST)
        return (np.array(m) > 127).astype(np.float32)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('L').resize((self.img_size, self.img_size))
        arr = np.array(img).astype(np.float32) / 255.0
        if self.mask_mode != 'none':
            arr = arr * self._mask(row)   # giu vung phoi, tat nen ngoai
        x = np.stack([arr] * 3, axis=0)    # grayscale -> 3 ch, CHW
        x = self.norm(torch.from_numpy(x))
        y = torch.tensor([row[c] for c in LABELS], dtype=torch.float32)
        return x, y

In [ ]:
# Kiem tra nhanh: decode 1 mask + overlay len anh (doi chieu train/val/test)
row = te_df.iloc[0]
h, w = int(row['Height']), int(row['Width'])
lung = decode_rle(row['Left Lung'], h, w) | decode_rle(row['Right Lung'], h, w)
img = np.array(Image.open(row['path']).convert('L').resize((IMG_SIZE, IMG_SIZE)))
m = np.array(Image.fromarray((lung * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)) > 127
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(img, cmap='gray'); ax[0].set_title(row['image_index']); ax[0].axis('off')
ax[1].imshow(m, cmap='gray');   ax[1].set_title('lung mask');       ax[1].axis('off')
ax[2].imshow(img * m, cmap='gray'); ax[2].set_title('masked');      ax[2].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# pos_weight = N_neg / N_pos per label (tu train), clamp tranh cuc dai
pos = tr_df[LABELS].sum().clip(lower=1)
neg = len(tr_df) - tr_df[LABELS].sum()
pos_weight = torch.tensor((neg / pos).clip(upper=20).values, dtype=torch.float32).to(device)
print('pos_weight (5 dau):', [round(v, 2) for v in pos_weight[:5].tolist()])

train_ds = CheXSparkDataset(tr_df, IMG_SIZE, MASK_MODE)
val_ds   = CheXSparkDataset(va_df, IMG_SIZE, MASK_MODE)
test_ds  = CheXSparkDataset(te_df, IMG_SIZE, MASK_MODE)
train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True)
val_dl   = DataLoader(val_ds, BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS)
test_dl  = DataLoader(test_ds, BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS)
print('DataLoaders ready.')

In [ ]:
def build_model(n_classes):
    m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    m.classifier = nn.Linear(m.classifier.in_features, n_classes)
    return m

model = build_model(len(LABELS)).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

@torch.no_grad()
def evaluate(dl):
    model.eval()
    ys, ps = [], []
    for x, y in dl:
        p = torch.sigmoid(model(x.to(device)))
        ys.append(y.numpy()); ps.append(p.cpu().numpy())
    Y, P = np.concatenate(ys), np.concatenate(ps)
    aucs = []
    for i in range(Y.shape[1]):
        if 0 < Y[:, i].sum() < len(Y):
            aucs.append(roc_auc_score(Y[:, i], P[:, i]))
        else:
            aucs.append(float('nan'))  # label degenerate trong tap nay (toan 0 hoac toan 1)
    return float(np.nanmean(aucs)), aucs
print('Model + loss + eval ready.')

In [ ]:
best = 0.0
history = {'train_loss': [], 'val_auc': []}
for ep in range(EPOCHS):
    model.train()
    t0, tot = time.time(), 0.0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step()
        tot += loss.item() * len(x)
    scheduler.step()
    tr_loss = tot / len(train_ds)
    va_auc, _ = evaluate(val_dl)
    history['train_loss'].append(tr_loss); history['val_auc'].append(va_auc)
    print(f'Ep{ep+1:02d}/{EPOCHS}  loss={tr_loss:.4f}  val_mAU={va_auc:.4f}  ({time.time()-t0:.0f}s)')
    if va_auc > best:
        best = va_auc
        torch.save(model.state_dict(), 'best_stage2_densenet121.pth')
print('Best val mAU:', round(best, 4))

In [ ]:
model.load_state_dict(torch.load('best_stage2_densenet121.pth'))
test_auc, aucs = evaluate(test_dl)
print(f'TEST mean AUC = {test_auc:.4f}')
res = pd.DataFrame({'label': [c.replace('label_', '') for c in LABELS], 'auc': aucs})
res = res.sort_values('auc', ascending=False)
print(res.to_string(index=False))

In [ ]:
# Luu model + metrics (working dir cua Kaggle Notebook Output)
torch.save(model.state_dict(), 'stage2_densenet121_final.pth')
with open('stage2_metrics.json', 'w') as f:
    json.dump({
        'best_val_auc': best,
        'test_auc': test_auc,
        'per_label_auc': {k: (None if pd.isna(v) else float(v)) for k, v in zip(res.label, res.auc)},
        'config': {'img_size': IMG_SIZE, 'epochs': EPOCHS, 'lr': LR, 'mask_mode': MASK_MODE},
    }, f, indent=2)
print('Da luu stage2_densenet121_final.pth + stage2_metrics.json.')
print('De giu lai: xem Kernel Output, hoac push thanh Kaggle Dataset.')